# ¿Cuándo se apagó la cámara térmica?

**Tau Labs · Clase 01 · Métodos Numéricos**

Durante una prueba, una cámara mantiene un bloque de aluminio a 80.0 °C. Un corte apaga la
cámara y reinicia el registrador; cuando vuelve a las 14:00, el bloque ya se está enfriando.
Conservamos 24 mediciones posteriores y la temperatura de la sala. ¿Podemos reconstruir a qué
hora ocurrió el corte? Es un problema inverso: observamos el efecto y queremos inferir el
evento que lo produjo.

> **Antes del código:** si dos modelos explicaran casi igual los datos posteriores, ¿qué
> tendrías que revisar antes de confiar en la hora reconstruida?

El dataset (`data/bloque_termico.csv`) es de demostración: `T(t) = 22.4 + 32.2·exp(-0.0068·t)`
con perturbaciones deterministas del orden de ±0.1 °C. No es una medición experimental real.

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/bloque_termico.csv")
T_sala = 22.4
df.head(6)

## Modelo

Ley de enfriamiento de Newton, de primer orden: la velocidad de cambio es proporcional a la
diferencia de temperatura entre el bloque y la sala.

$$\frac{dT}{dt} = -k\,(T - T_{sala}) \quad\Rightarrow\quad T(t) = T_{sala} + A\,e^{-kt}$$

**Supuestos:** bloque aproximadamente isotermo · k constante · sala representada por 22.4 °C ·
sin calentamiento activo tras el corte.

**Lo que hay que decidir:** qué error atribuimos al sensor, si ajustamos temperaturas o una
transformación logarítmica, y hasta dónde es razonable extrapolar fuera de los datos.

In [ ]:
from scipy.optimize import curve_fit

def T(t, A, k):
    return T_sala + A * np.exp(-k * t)

(A, k), cov = curve_fit(T, df.t_min, df.T_bloque_C, p0=(32, 0.007))

resid = df.T_bloque_C - T(df.t_min, A, k)
ss_res = np.sum(resid**2)
ss_tot = np.sum((df.T_bloque_C - df.T_bloque_C.mean())**2)
r2 = 1 - ss_res / ss_tot
print(f"A = {A:.3f} °C   k = {k:.6f} min^-1")
print(f"R² = {r2:.5f}   max|residuo| = {np.abs(resid).max():.3f} °C")

## Solución numérica

La cámara estaba estabilizada en 80.0 °C antes del corte. Buscamos el tiempo $t<0$ en que el
modelo alcanza esa temperatura. Podríamos despejar; usamos bisección para estudiar un patrón
reutilizable y reservamos el despeje como verificación independiente.

In [ ]:
def g(t):
    return T(t, A, k) - 80.0

a, b = -120.0, -60.0
assert g(a) * g(b) < 0, "el intervalo no encierra la raíz"
for n in range(20):
    m = (a + b) / 2
    if g(a) * g(m) <= 0:
        b = m
    else:
        a = m
    if n % 4 == 0 or n == 19:
        print(f"{n:2d}  [{a:10.4f}, {b:10.4f}]  medio = {m:.4f}")

t_corte = (a + b) / 2
# verificación analítica
t_exact = -np.log((80.0 - T_sala) / A) / k
print(f"\nbisección: {t_corte:.3f} min   despeje: {t_exact:.3f} min")

In [ ]:
from datetime import datetime, timedelta

t0 = datetime(2026, 1, 1, 14, 0, 0)          # primera medición
hora_corte = t0 + timedelta(minutes=float(t_corte))
print(f"El corte ocurrió ≈ {abs(t_corte):.2f} min antes del reinicio: {hora_corte:%H:%M:%S}")

## Validación

La precisión algorítmica no es la incertidumbre del problema. Variamos el ajuste, la temperatura
de sala y la ventana de datos para ver qué parte de la inferencia es realmente estable.

In [ ]:
def reconstruir(T_sala_val, mask=None):
    d = df if mask is None else df[mask]
    f = lambda t, A, k: T_sala_val + A * np.exp(-k * t)
    (A_, k_), _ = curve_fit(f, d.t_min, d.T_bloque_C, p0=(32, 0.007))
    return -np.log((80.0 - T_sala_val) / A_) / k_

sigma = np.sqrt(np.diag(cov))
print(f"ajuste (95%):     {reconstruir(T_sala):.1f} min  ± param.")
print(f"T_sala 21.9 / 22.9: {reconstruir(21.9):.1f} / {reconstruir(22.9):.1f} min")
print(f"primeros 120 min:  {reconstruir(T_sala, df.t_min <= 120):.1f} min")

La mayor sensibilidad está en `T_sala`: ±0.5 °C mueve la inferencia casi ±2 min, mucho más que la
incertidumbre estadística del ajuste. El algoritmo converge; el modelo domina la confianza.

## Transferencia

La misma estructura — decaimiento exponencial hacia un equilibrio, parámetro ajustado con datos,
pregunta resuelta como raíz — aparece en otros contextos:

| Dominio | Sistema | Pregunta |
|---|---|---|
| Circuitos | Descarga RC | ¿Cuándo cruzó 1.8 V? |
| Química | Cinética de primer orden | ¿Cuándo quedó 10%? |
| Control | Respuesta de primer orden | ¿Cuándo entró en ±2%? |

**Evaluación:** resolver uno de estos contextos con datos nuevos y justificar modelo, método,
validación y límites.